# Example: Roomba Lava World Task using Q-Learning
In this example, we will solve a grid-world navigation task using Q-learning. Here is the scenario:

__Scenario__: Suppose you have a robot vacuum cleaner, e.g., a [roomba](https://www.irobot.com), that has finished cleaning the kitchen floor and needs to return to its charging station. However, between your kitchen floor and the charging station (safety), there are one or more lava pits (destruction for the [roomba](https://www.irobot.com)). This two-dimensional grid-world navigation task (which is similar to the Markov Decision Process problem that we explored earlier) will familiarize students with using Q-learning for solving a sequential decision task in which we have __no model of the world!__ 

How does the robot learn to navigate safely back to its charging station while avoiding the lava pits? The answer is through trial and error using a model-free reinforcement learning algorithm.

> __Learning Objectives:__
> 
> After completing this activity, students will be able to:
>
> * __Understand Q-learning:__ We explore the theoretical foundations of Q-learning. This model-free reinforcement learning algorithm learns optimal policies through trial and error by updating action-value estimates based on observed rewards.
> * __Implement Q-learning:__ We build and execute a Q-learning agent that learns to navigate a grid world environment. The implementation shows how to construct the environment model, define reward structures, and train the agent through repeated episodes.
> * __Apply reward shaping techniques:__ We use radial basis functions to shape sparse rewards, providing the agent with guidance signals that accelerate learning without requiring full knowledge of the optimal policy.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating project at `~/Desktop/julia_work/CHEME-150-eCornell-Repository/CHEME-150-eCornell-Repository/courses/CHEME-153/module-4`


In addition to standard Julia libraries, this example uses local source files for the types and functions it needs: the model types are defined in [`src/Types.jl`](src/Types.jl), their constructors in [`src/Factory.jl`](src/Factory.jl), and the Q-learning routines `solve(...)` and `policy(...)` in [`src/Compute.jl`](src/Compute.jl). These are loaded for you by `Include.jl`.

### Implementation
We implement two helper functions for the Q-learning algorithm. The `rbf(...)` function computes the radial basis function kernel, which measures similarity between two grid positions. The `lavaworld(...)` function defines the environment dynamics, taking the current state and action as inputs and returning the next state and reward.

In [ ]:
function rbf(x::Tuple{Int,Int},y::Tuple{Int,Int}; σ = 1.0)::Float64
    d2 = (x[1] - y[1])^2 + (x[2] - y[2])^2; # squared Euclidean distance
    return exp(-d2/(2*σ^2))
end;

The `lavaworld(...)` function simulates one step in the environment. Given the current state `s::Int` and action `a::Int`, it computes the next state `s′::Int` and the reward `r::Float64`. If the agent moves off the grid, it stays in place and receives a large negative penalty.

In [ ]:
function lavaworld(environment::MyRectangularGridWorldModel, s::Int, a::Int)

    # initialize -
    s′ = nothing
    r = nothing
    coordinates = environment.coordinates;
    moves = environment.moves
    states = environment.states;
    rewards = environment.rewards;

    # where are we now?
    current_position = coordinates[s];

    # get the perturbation -
    Δ = moves[a];
    new_position = current_position .+ Δ

    # before we go on, have we "driven off the grid"?
    if (haskey(states, new_position) == true)

        # lookup the new state -
        s′ = states[new_position];
        r = rewards[s′];
    else
       
        # ok: so we are off the grid. Bounce us back to the current_position, and charge a huge penalty 
        s′ = states[current_position];
        r = -1000000000000.0
    end

    # return -
    return (s′,r);
end

___

## Task 1: Build the Rectangular Grid World Model
In this task, we set up the grid world model that our navigation agent lives in. We know this world, but the agent must discover it through trial and error. We encode the rectangular grid world using [the `MyRectangularGridWorldModel` type](src/Types.jl), which represents a spatial environment where a robot must navigate from any starting position to a charging station while avoiding lava pits. 

> __Grid World Structure:__ The model consists of:
> * A 30×30 grid with $(x,y)$ coordinates mapping to discrete states
> * Positive rewards for reaching the charging station (goal)
> * Large negative penalties for lava pits (hazards)
> * Small step costs for regular movements (efficiency incentive)
> * Charging station (goal) and lava pits (hazards), which mark the cells where a trajectory is considered finished

Let's set up the world parameters by defining the grid dimensions, the number of available actions (up, down, left, right), and the discount factor $\gamma$ that controls how much we value future rewards relative to immediate rewards:

First, set values for the `number_of_rows::Int` and `number_of_cols::Int` variables, the `nactions::Int` available to the agent, and the discount factor `γ::Float64`. Then, we compute the number of states `nstates::Int` and set up the state set `𝒮::Vector{Int}` and the action set `𝒜::Vector{Int}`

In [ ]:
number_of_rows = 30; # number of rows in the grid world (you can change this)
number_of_cols = 30; # number of cols in the grid world (you can change this)
nactions = 4; # number of actions (up, down, left, right)
γ = 0.95; # discount factor (you can change this)
nstates = (number_of_rows*number_of_cols);
𝒮 = range(1,stop=nstates,step=1) |> collect; # states
𝒜 = range(1,stop=nactions,step=1) |> collect; # actions

Next, we define the reward structure by creating the `rewards::Dict{Tuple{Int,Int}, Float64}` dictionary, which maps $(x,y)$ coordinates to reward values. We set the `lava_reward::Float64` to $-1000$ for dangerous locations that destroy the robot, while the `charging_reward::Float64` is $+100$ for successfully reaching the goal. We also define an `absorbing_state_set::Set{Tuple{Int,Int}}` containing locations where the episode terminates. Note that we only specify non-default rewards in this dictionary since `default_reward::Float64` step costs will be added later. 

Finally, we add some softwall locations to the `soft_wall_set::Set{Tuple{Int,Int}}`, which represent areas that are difficult to traverse but not completely blocked, e.g. through the legs of a kitchen table.

In [5]:
rewards, soft_wall_set, absorbing_state_set, charging_reward, lava_reward, default_reward = let

    # initialize -
    lava_reward = -1000.0; # lava pit squares, huge negative reward
    charging_reward = 100.0; # charging station square, positive reward
    softwall_reward = -200.0;  # soft wall squares, negative reward
    default_reward = -1.0; # default step cost (costs power to move around)

    # setup rewards -
    rewards = Dict{Tuple{Int,Int}, Float64}()
    rewards[(5,5)] =  lava_reward # lava in the (5,5) square 
    rewards[(6,5)] = lava_reward # lava in the (6,5) square
    rewards[(6,6)] = lava_reward # lava in the (6,6) square
    rewards[(5,6)] = charging_reward    # charging station square

    # Non-lava pit obstacles (soft walls) -
    soft_wall_set = Set{Tuple{Int,Int}}();
    push!(soft_wall_set, (2,1));
    push!(soft_wall_set, (2,2));
    push!(soft_wall_set, (2,3));
    push!(soft_wall_set, (7,4));
    push!(soft_wall_set, (4,6));
    for s in soft_wall_set
        rewards[s] = softwall_reward;
    end

    # setup set of absorbing states -
    absorbing_state_set = Set{Tuple{Int,Int}}()
    for (k,v) ∈ rewards
        push!(absorbing_state_set, k);   
    end

    # return
    rewards, soft_wall_set, absorbing_state_set, charging_reward, lava_reward, default_reward
end

(Dict((5, 5) => -1000.0, (7, 4) => -200.0, (6, 5) => -1000.0, (2, 2) => -200.0, (5, 6) => 100.0, (2, 1) => -200.0, (6, 6) => -1000.0, (4, 6) => -200.0, (2, 3) => -200.0), Set([(7, 4), (2, 2), (2, 1), (4, 6), (2, 3)]), Set([(5, 5), (7, 4), (6, 5), (2, 2), (5, 6), (2, 1), (6, 6), (4, 6), (2, 3)]), 100.0, -1000.0, -1.0)

### Reward shaping
In cases where rewards are sparse, there are few $(s,a)$ pairs that lead to non-zero rewards. This is an issue because reinforcement learning algorithms, e.g., Q-learning behave randomly initially when the action-value function $Q(s,a)$ is unknown. They can spend a long time exploring the environment without ever encountering a reward signal, making it difficult to learn an effective policy. We can address this issue using reward shaping.

> __Reward Shaping__
> 
> [Reward shaping](https://gibberblot.github.io/rl-notes/single-agent/reward-shaping.html) is an approach to address this issue, by modifying the reward function to promote behavior that we think will move us closer to the goal state, e.g., the `charging_station`. There are different approaches to this. We'll use a [radial basis kernel function](https://en.wikipedia.org/wiki/Radial_basis_function_kernel) to distribute charging station rewards radially from the goal state.

What does this look like in real life? Maybe the charging station is sending out a weak signal that the robot can detect from a distance, and the strength of this signal increases as the robot gets closer to the charging station. This way, even if the robot is far away from the charging station and cannot directly reach it, it can still receive some positive feedback that encourages it to move in the right direction.

We'll update the `rewards::Dict{Tuple{Int,Int}, Float64}` dictionary to include these shaped rewards if the `is_reward_shaping_on::Bool` flag is set to `true`. The `σ::Float64` parameter controls the width of the radial basis function. 

In [ ]:
# do some shaping?
is_reward_shaping_on = true;
σ = 2.0; # width parameter for rbf
if (is_reward_shaping_on == true)
    for i in 1:number_of_rows
        for j in 1:number_of_cols
            coordinate = (i,j);
            # only shape "empty" cells; lava, charging, and soft-wall cells are already keys in rewards
            if (haskey(rewards, coordinate) == false)
                rewards[coordinate] = default_reward + charging_reward*rbf(coordinate, (5,6), σ = σ);
            end
        end
    end
end

Finally, build an instance of the `MyRectangularGridWorldModel` type, which models the grid world. We save this instance in the `world_model::MyRectangularGridWorldModel` variable. Pass in the number of rows `nrows::Int`, number of cols `ncols::Int`, and our initial reward description in the `rewards::Dict{Tuple{Int,Int}, Float64}` field into the `build(...)` method

In [ ]:
# call the factory -
world_model = build(MyRectangularGridWorldModel, (
        nrows=number_of_rows, ncols=number_of_cols, rewards = rewards, defaultreward = default_reward));

UndefVarError: UndefVarError: `VLDataScienceMachineLearningPackage` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___

## Task 2: Build the Q-learning agent
In this task, we build a Q-learning agent that will learn to navigate the grid world we defined in Task 1.
We build an `agent_model::MyQLearningAgentModel`, i.e., the code representation of the [roomba](https://www.irobot.com) using the `MyQLearningAgentModel` type encoded in the [Types.jl file](src/Types.jl) and an associated `build` function encoded in the [Factory.jl file](src/Factory.jl).

The learning rate parameter `α::Float64` describes the weight of the update step (how we incorporate new information), while the discount rate `γ::Float64` is the weight of future steps. In this example, we initialize `Q::Matrix{Float64}` to a matrix of zeros.

In [8]:
agent_model = let

    # initialize -
    α = 0.8;  # learning rate
    γ = 0.95; # discount rate
    nstates = (number_of_rows*number_of_cols);
    agent_model = build(MyQLearningAgentModel, (
        states = 𝒮,
        actions = 𝒜,
        α = α,
        γ = γ,
        Q = zeros(nstates,nactions) # initialize Q-table to zeros
    ));

    agent_model; # return model -
end

MyQLearningAgentModel([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  891, 892, 893, 894, 895, 896, 897, 898, 899, 900], [1, 2, 3, 4], 0.95, 0.8, [0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0; … ; 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0])

___

## Task 3: Simulate and visualize
To begin, set the `startstate::Tuple{Int,Int}` variable. Training sweeps over every state, so `startstate` is used only by the visualization below, which traces the learned greedy path starting from this cell:

In [9]:
startstate = (30,30); # start position

Next, we train the agent using the `solve(...)` function. The function takes the `agent_model::MyQLearningAgentModel` and `world_model::MyRectangularGridWorldModel` as inputs, along with keyword arguments for maximum steps per episode (`maxsteps::Int`), convergence threshold `δ::Float64`, and the environment dynamics function. The function returns a `result` containing the learned `Q::Matrix{Float64}` table.

In [ ]:
result = solve(agent_model, world_model; 
    maxsteps = 100, δ = 0.02, worldmodel = lavaworld);

UndefVarError: UndefVarError: `VLDataScienceMachineLearningPackage` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

The learned Q-table is a $900 \times 4$ matrix where each row represents a state (grid position) and each column represents an action (up, down, left, right). Each entry $Q(s,a)$ estimates the expected cumulative reward from taking action $a$ in state $s$ and following the optimal policy thereafter:

In [11]:
result.Q

UndefVarError: UndefVarError: `result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

We can now build the policy $\pi$ function from the `Q::Matrix{Float64}` table. We iterate through each state $s\in\mathcal{S}$ and select the action $a\in\mathcal{A}$ that gives the highest utility. We've implemented this logic in the [policy function](src/Compute.jl), which takes the `Q::Matrix{Float64}` table as its only argument and returns `my_π::Vector{Int}`, a vector mapping each state index to an action.

In [12]:
my_π = policy(result.Q);

UndefVarError: UndefVarError: `result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

`Unhide` the code block below to see how we plot the path through our grid world that [roomba](https://www.irobot.com/en_US/roomba.html?source=google_paid&medium=cpc&ds_campaign=US+-+Conversion+-+Brand+-+SEM+-+Roomba+-+Core+-+Exact&ds_content=Core+-+Roomba+-+Exact&ds_keyword=roomba&gad_source=1&gclid=EAIaIQobChMIu_65qsfYhQMVImJHAR3i1wTGEAAYASAAEgKwgfD_BwE&gclsrc=aw.ds) uses to get back to the charging station (shown as the green circle) while navigating around the lava pits (shown as red circles).

In [13]:
let

    # initialize -
    p = plot();
    initial_site = startstate
    hit_absorbing_state = false
    s = world_model.states[initial_site];
    visited_sites = Set{Tuple{Int,Int}}();
    push!(visited_sites, initial_site);

    while (hit_absorbing_state == false)
        current_position = world_model.coordinates[s]
        a = my_π[s];
        Δ = world_model.moves[a];
        new_position =  current_position .+ Δ
        scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, msc=:black, c=:blue)
        plot!([current_position[1], new_position[1]],[current_position[2],new_position[2]], label="", arrow=true, lw=1, c=:gray)
        
        if (in(new_position, absorbing_state_set) == true || in(new_position, visited_sites) == true)
            hit_absorbing_state = true;
        elseif (haskey(world_model.states, new_position) == true)
            s = world_model.states[new_position];
            push!(visited_sites, new_position);
        else
            hit_absorbing_state = true; # we drove off the map
        end
    end

    # draw the grid -
    for s ∈ 𝒮
        current_position = world_model.coordinates[s]
        a = my_π[s];
        Δ = world_model.moves[a];
        new_position =  current_position .+ Δ
        
        if (haskey(rewards, current_position) == true && rewards[current_position] == charging_reward)
            scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, c=:green, ms=4)
        elseif (haskey(rewards, current_position) == true && rewards[current_position] == lava_reward)
            scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, c=:red, ms=4)
        elseif (in(current_position, soft_wall_set) == true)
            scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, c=:gray69, ms=4)
        else
            if (is_reward_shaping_on == true)
                new_color = weighted_color_mean(rbf(current_position, (5,6), σ = σ), colorant"green", colorant"white")
                scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, msc=:lightgray, c=new_color)
            else
                scatter!([current_position[1]],[current_position[2]], label="", showaxis=:false, msc=:black, c=:white)
            end
        end
    end
    current()
end

UndefVarError: UndefVarError: `world_model` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

___

## Summary
In this activity, we used Q-learning to train a robot to navigate a grid world with obstacles, rewards, and penalties.

> __Key Takeaways:__
>
> * __Model-free learning:__ Q-learning does not require knowledge of the environment dynamics. The agent learns by interacting with the environment and observing rewards, making it suitable for problems where the transition model is unknown.
> * __Action-value function:__ The algorithm builds a Q-table that estimates the expected return for each state-action pair. The policy is derived by selecting the action with the highest Q-value in each state.
> * __Reward shaping accelerates learning:__ In environments with sparse rewards, shaping techniques like radial basis functions provide additional guidance signals that help the agent discover productive behaviors more quickly.

Q-learning enables agents to learn optimal policies through experience without requiring a model of the environment.
___